# Modeling Daily Rat Sightings in New York City

In [415]:
# import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error


from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

## Importing the Data

In [416]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

In [417]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])

In [418]:
# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

In [419]:
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')

## Baseline Seasonal Average Model

In [420]:
years_back_use = 4
day_window_use = 4

In [421]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    # ensure datetime type
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["doy"] = df["created_date"].dt.dayofyear
    df["year"] = df["created_date"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "count"].mean())
    return pd.Series(forecasts, index=target_dates)

In [422]:
results = []

rs["created_date"] = pd.to_datetime(rs["created_date"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["created_date"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)

    # We take the true values.
    y_true = test["count"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,15.307003,0.285977
1,1,11.914386,0.186347
2,2,14.988228,0.250230
3,3,20.818662,0.348457
4,4,17.563835,0.256820
5,5,23.244116,0.361006
6,6,19.488233,0.266129
7,7,21.222927,0.284166
8,8,20.494910,0.301036
9,9,21.193296,0.277230


## Year Ago Rolling 4 Week Average 

In [423]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

# Optional: add a row for the overall average RMSE and MAPE
overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [424]:
rolling4w_results_df

,fold,rmse,mape
0,0,14.323744,0.239272
1,1,12.719150,0.196755
2,2,12.398157,0.196394
3,3,19.240721,0.260374
4,4,15.923646,0.220183
5,5,17.378918,0.256428
6,6,24.564601,0.247419
7,7,20.989793,0.255484
8,8,19.858428,0.253080
9,9,18.798509,0.187194


## Prophet Model

In [425]:
# Create a date range covering 2020 through end of 2025
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

In [426]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

23:29:22 - cmdstanpy - INFO - Chain [1] start processing
23:29:22 - cmdstanpy - INFO - Chain [1] done processing
23:29:22 - cmdstanpy - INFO - Chain [1] start processing
23:29:22 - cmdstanpy - INFO - Chain [1] done processing
23:29:23 - cmdstanpy - INFO - Chain [1] start processing
23:29:23 - cmdstanpy - INFO - Chain [1] done processing
23:29:23 - cmdstanpy - INFO - Chain [1] start processing
23:29:23 - cmdstanpy - INFO - Chain [1] done processing
23:29:24 - cmdstanpy - INFO - Chain [1] start processing
23:29:24 - cmdstanpy - INFO - Chain [1] done processing
23:29:25 - cmdstanpy - INFO - Chain [1] start processing
23:29:25 - cmdstanpy - INFO - Chain [1] done processing
23:29:25 - cmdstanpy - INFO - Chain [1] start processing
23:29:25 - cmdstanpy - INFO - Chain [1] done processing
23:29:26 - cmdstanpy - INFO - Chain [1] start processing
23:29:26 - cmdstanpy - INFO - Chain [1] done processing
23:29:26 - cmdstanpy - INFO - Chain [1] start processing
23:29:26 - cmdstanpy - INFO - Chain [1]

In [427]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

In [428]:
prophet_results_df

,fold,rmse,mape
0,0,12.931175,0.246172
1,1,9.286185,0.142636
2,2,7.983666,0.130110
3,3,9.817654,0.143238
4,4,10.903003,0.138791
5,5,15.534582,0.217436
6,6,11.539387,0.145670
7,7,11.299325,0.135217
8,8,10.559222,0.118595
9,9,9.146496,0.113947


## SARIMA Model with auto_arima parameters.

In [429]:
pip install pmdarima

Note: you may need to restart the kernel to use updated packages.


In [430]:
from pmdarima import auto_arima

In [431]:
def fourier_terms(df, period, n_terms):
    t = np.arange(1, len(df) + 1)
    fourier_df = pd.DataFrame()
    
    for i in range(1, n_terms + 1):
        fourier_df[f'sin_{i}'] = np.sin(2 * np.pi * i * t / period)
        fourier_df[f'cos_{i}'] = np.cos(2 * np.pi * i * t / period)
    
    return fourier_df

In [432]:
# Number of Fourier terms and period (365 for yearly seasonality)
n_terms = 5  # Number of terms for Fourier Terms
period = 365 

In [433]:
# Generate Fourier terms and putting it into a list
fourier_train = fourier_terms(rs, period, n_terms)

# Name it exog since it will serve as exogenous features for SARIMAX. This is the X.
exog = fourier_train

In [434]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Make sure the columns for SARIMA model are renamed.
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

results = []

# Loop through each fold
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    exog_train = exog.iloc[train_index]
    exog_test = exog.iloc[test_index]

    orders = (1,1,1)
    seasonal_orders = (0,0,0,7)
    # # We use auto_arima to get optimal (p, d, q) and (P, D, Q, s) parameters for SARIMAX
    # model_auto = auto_arima(train['y'], 
    #                         exog=exog_train,  # Exogenous Fourier terms for training data
    #                         seasonal=True, 
    #                         m=365,  # Yearly seasonality for daily data
    #                         trace=True, 
    #                         stepwise=True,  # Stepwise search to speed up
    #                         suppress_warnings=True, 
    #                         n_jobs=-1,  # Use all available cores for parallel processing
    #                         maxiter=200,  # Limit the number of iterations
    #                         max_p=3, 
    #                         max_q=3, 
    #                         max_P=2, 
    #                         max_Q=2, 
    #                         d=1, 
    #                         D=1)
    # orders = model_auto.order  # (p, d, q)
    # seasonal_orders = model_auto.seasonal_order  # (P, D, Q, s)
    
    # Fit the SARIMAX model with the exogenous features (Fourier terms)
    model_sarimax = SARIMAX(train['y'], 
                            order=orders,  
                            seasonal_order=seasonal_orders,  
                            exog=exog_train,  # Exogenous Fourier terms for training data
                            enforce_stationarity=False,
                            enforce_invertibility=False)
    
    model_fit = model_sarimax.fit(disp=False)
    
    # Predict for the test period. Have to remember to subtract 1 to get the correct index.
    y_pred = model_fit.predict(start=len(train), end=len(train)+len(test)-1, exog=exog_test, dynamic=False)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

sarima_results_df = pd.DataFrame(results)

In [435]:
sarima_results_df.loc['mean'] = ['mean',  sarima_results_df['rmse'].mean(), sarima_results_df['mape'].mean()]

In [436]:
sarima_results_df

,fold,rmse,mape
0,0,15.719609,0.240440
1,1,14.271212,0.199697
2,2,12.654641,0.201484
3,3,17.120726,0.263835
4,4,14.470259,0.202554
5,5,17.907910,0.264081
6,6,14.692676,0.176597
7,7,20.815400,0.250376
8,8,20.015043,0.267852
9,9,13.988056,0.166753


## Holt-Winters Model

In [437]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [438]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # First we fit the Holt-Winters Exponential Smoothing Model to the training data
    holt_winters = ExponentialSmoothing(train['y'], seasonal='add', seasonal_periods=365).fit(optimized=True)
    
    y_pred = holt_winters.forecast(len(test))
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

hw_results_df = pd.DataFrame(results)

In [439]:
hw_results_df.loc['mean'] = ['mean',  hw_results_df['rmse'].mean(), hw_results_df['mape'].mean()]

In [440]:
hw_results_df

,fold,rmse,mape
0,0,17.029947,0.279665
1,1,17.558940,0.255894
2,2,17.312295,0.297496
3,3,22.556060,0.358384
4,4,19.966872,0.268691
5,5,20.725658,0.310186
6,6,22.555124,0.270105
7,7,25.834040,0.311909
8,8,23.343693,0.306820
9,9,21.316698,0.274190


## XGBoost Model

The XGBoost model requires a bit more preparatory work. Our current dataframe rs is quite bare. We will need to add features for use.

In [441]:
import xgboost as xgb

In [442]:
rs

,ds,y
0,2020-01-01,17
1,2020-01-02,40
2,2020-01-03,41
3,2020-01-04,25
4,2020-01-05,17
...,...,...
2246,2026-02-24,26
2247,2026-02-25,30
2248,2026-02-26,40
2249,2026-02-27,38


### Adding Features to XGBoost

In [443]:
def create_features(df):
    """
    Create time series features based on time series index.
    """
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    
    # Ensure sorted by date
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
    return df


In [444]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-12-31"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd

else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [445]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary column
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [446]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format → recurring annually
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD → one specific date
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [447]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    # Convert start_date to datetime
    start_dt = pd.to_datetime(start_date)
    
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [448]:
# This must be run importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    return df

In [449]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)
rs

,y
ds,
2020-01-01,17
2020-01-02,40
2020-01-03,41
2020-01-04,25
2020-01-05,17
...,...
2026-02-24,26
2026-02-25,30
2026-02-26,40


In [450]:
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

In [451]:
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,apparent_temperature_min_lag300,apparent_temperature_min_lag330,apparent_temperature_min_lag360,apparent_temperature_min_lag365,apparent_temperature_min_lag730,is_federal_holiday,Trash_Law,New_Trash_Law,Rat_Mitigation_Zone,Rat_Czar_Appointed
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24,26,1,1,2,2026,55,24,9,0.781831,0.623490,...,12.1,7.0,-9.1,-4.6,-11.1,0,1,1,1,1
2026-02-25,30,2,1,2,2026,56,25,9,0.974928,-0.222521,...,7.1,-0.6,-13.5,0.1,-6.9,0,1,1,1,1
2026-02-26,40,3,1,2,2026,57,26,9,0.433884,-0.900969,...,12.9,-4.2,-10.2,-1.4,-4.1,0,1,1,1,1


### Features for XGBoost

In [452]:
FEATURES = ['lag7', 'lag14', 'lag90', 'lag120', 'lag180', 'lag362',
            'apparent_temperature_min_lag7',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag150',
            'apparent_temperature_min_lag180',
            'apparent_temperature_min_lag210',
            'dayofyear', 'temperature_2m_max', 
            ]

### Add default parameters for XGBoost

In [453]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': 1, 
         'n_estimators': 1000, 
        #  'early_stopping_rounds': 100, 
         'min_child_weight': 6, 
         'learning_rate': 0.001,
         'max_depth': 4, 
         'subsample': 1,
         'colsample_bytree': 0.96,
         'colsample_bylevel': 0.6, 
         'colsample_bynode': 0.9, 
         'reg_alpha': 2.2, 
         'gamma': 100, 
         'reg_lambda': 0.18}


In [454]:
print(FEATURES)
print(params)
TARGET = 'y'

# Gotta make sure the features and parameters exist.

reg = xgb.XGBRegressor(**params)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    reg.fit(train[FEATURES], train[TARGET])
    y_pred = reg.predict(test[FEATURES])
    y_true = test[TARGET].values
    
    # Our metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

xgb_results_df = pd.DataFrame(results)
mean_rmse = xgb_results_df['rmse'].mean()
mean_mape = xgb_results_df['mape'].mean()
xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

['lag7', 'lag14', 'lag90', 'lag120', 'lag180', 'lag362', 'apparent_temperature_min_lag7', 'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730', 'apparent_temperature_min_lag30', 'apparent_temperature_min_lag60', 'apparent_temperature_min_lag120', 'apparent_temperature_min_lag150', 'apparent_temperature_min_lag180', 'apparent_temperature_min_lag210', 'dayofyear', 'temperature_2m_max']
{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'booster': 'gbtree', 'base_score': 1, 'n_estimators': 1000, 'min_child_weight': 6, 'learning_rate': 0.001, 'max_depth': 4, 'subsample': 1, 'colsample_bytree': 0.96, 'colsample_bylevel': 0.6, 'colsample_bynode': 0.9, 'reg_alpha': 2.2, 'gamma': 100, 'reg_lambda': 0.18}


In [455]:
xgb_results_df

,fold,rmse,mape
0,0,24.356770,0.342589
1,1,23.109263,0.317701
2,2,21.757512,0.327479
3,3,25.602124,0.326805
4,4,26.805798,0.335786
5,5,23.389577,0.268785
6,6,30.755804,0.352544
7,7,30.409906,0.316881
8,8,27.277671,0.334818
9,9,25.649643,0.321863


# Conclusions

In [456]:
# We make a dictionary of models and their results to make it easier to iterate over.
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
    'prophet': prophet_results_df,
    'sarima': sarima_results_df,
    'hw': hw_results_df,
    'xgb': xgb_results_df
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)

# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                         \
model   baseline         hw    prophet  rolling4w     sarima        xgb   
fold                                                                      
0      15.307003  17.029947  12.931175  14.323744  15.719609  24.356770   
1      11.914386  17.558940   9.286185  12.719150  14.271212  23.109263   
2      14.988228  17.312295   7.983666  12.398157  12.654641  21.757512   
3      20.818662  22.556060   9.817654  19.240721  17.120726  25.602124   
4      17.563835  19.966872  10.903003  15.923646  14.470259  26.805798   
5      23.244116  20.725658  15.534582  17.378918  17.907910  23.389577   
6      19.488233  22.555124  11.539387  24.564601  14.692676  30.755804   
7      21.222927  25.834040  11.299325  20.989793  20.815400  30.409906   
8      20.494910  23.343693  10.559222  19.858428  20.015043  27.277671   
9      21.193296  21.316698   9.146496  18.798509  13.988056  25.649643   
10     31.447761  23.475082  15.403790  15.939618  17.332377  22.182639   
11     22.102668  20.733607   9.164685  13.952726  14.142204  25.791463   
12     23.922867  18.607673  11.141143  17.455044  16.634764  27.306057   
13     19.713532  26.490924  12.065524  19.822201  19.536495  30.392571   
14     34.114645  26.457852  18.946567  21.751026  20.134967  15.538995   
15     26.451596  17.653667   8.367090  13.761683  14.513849  19.743732   
16     28.910678  15.214237  11.798670  12.954591  10.708175  10.427355   
17     35.881458  16.538625  15.583406  19.309278  15.595443  11.336222   
18     24.802787  13.508845  11.766219   8.641676  10.274787  11.449313   
19     18.947508  17.519674  11.907598  15.194865  14.260241  14.811382   
20     24.849791  11.076713  10.669815  14.811916   7.116006   6.853505   
21     22.731809  12.828411   9.164783   8.280248   9.270426   7.205611   
22     20.216302  13.621282   9.808565  15.873383  13.158872  17.119646   
23     30.933127  15.432451  11.468369  11.649647  12.523939   8.699722   
24     30.349771   6.899231   9.018752   8.625129   7.722993   6.460631   
25     26.045942  14.060919  14.222347  13.177023  10.310420  15.613014   
mean   23.371455  18.396866  11.519154  15.669066  14.418903  19.617151   

           mape                                                    
model  baseline        hw   prophet rolling4w    sarima       xgb  
fold                                                               
0      0.285977  0.279665  0.246172  0.239272  0.240440  0.342589  
1      0.186347  0.255894  0.142636  0.196755  0.199697  0.317701  
2      0.250230  0.297496  0.130110  0.196394  0.201484  0.327479  
3      0.348457  0.358384  0.143238  0.260374  0.263835  0.326805  
4      0.256820  0.268691  0.138791  0.220183  0.202554  0.335786  
5      0.361006  0.310186  0.217436  0.256428  0.264081  0.268785  
6      0.266129  0.270105  0.145670  0.247419  0.176597  0.352544  
7      0.284166  0.311909  0.135217  0.255484  0.250376  0.316881  
8      0.301036  0.306820  0.118595  0.253080  0.267852  0.334818  
9      0.277230  0.274190  0.113947  0.187194  0.166753  0.321863  
10     0.498262  0.358957  0.226627  0.229220  0.259642  0.274387  
11     0.317831  0.281012  0.116819  0.194994  0.191667  0.324869  
12     0.359283  0.213410  0.119338  0.200704  0.184555  0.300607  
13     0.275826  0.314054  0.120530  0.223940  0.211381  0.341105  
14     0.766270  0.575931  0.387667  0.478330  0.431906  0.261158  
15     0.582483  0.338866  0.157208  0.290997  0.297773  0.302666  
16     0.723483  0.350417  0.256426  0.308417  0.249026  0.175790  
17     1.485229  0.650859  0.553277  0.808842  0.639386  0.329170  
18     0.767713  0.299738  0.308413  0.170184  0.201983  0.220421  
19     0.787489  0.500986  0.365745  0.628196  0.383895  0.387419  
20     1.259635  0.512212  0.409370  0.750920  0.315849  0.247511  
21     1.276241  0.573143  0.405596  0.466834  0.522615  0.394300  
22     0.645109  0.367359  0.241197  0.318148  0.290013  0.35